<a href="https://colab.research.google.com/github/ahmadd4062/AI-Interview-Insight/blob/main/gait%20model%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
print("="*60)
print("INSTALLING REQUIRED LIBRARIES")
print("="*60)

# Install TensorFlow and other dependencies
!pip install tensorflow scikit-learn opencv-python mediapipe -q

# Verify installation
import tensorflow as tf
import mediapipe as mp
import cv2
import numpy as np

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ MediaPipe version: {mp.__version__}")
print("✅ All libraries installed successfully!")

INSTALLING REQUIRED LIBRARIES
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.5/36.5 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 10.2 MB/s eta 0:00:00
✅ TensorFlow version: 2.20.0
✅ MediaPipe version: 1.0.0
✅ All libraries installed successfully!


In [12]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from scipy.signal import find_peaks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

print("✅ Setup complete!")

PROJECT_PATH = "/content/drive/MyDrive/NeuroSense AI 2/"
DATASET_PATH = PROJECT_PATH + "MMU Dataset/"
MODELS_PATH = PROJECT_PATH + "models mmu/"

os.makedirs(MODELS_PATH, exist_ok=True)

print(f"Dataset: {DATASET_PATH}")
print(f"Models: {MODELS_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete!
Dataset: /content/drive/MyDrive/NeuroSense AI 2/MMU Dataset/
Models: /content/drive/MyDrive/NeuroSense AI 2/models mmu/


In [13]:
print("="*60)
print("EXTRACTING POSE LANDMARKS (33 KEYPOINTS) FROM MMU")
print("="*60)

def extract_pose_sequence(json_path, sequence_length=50):
    """
    Extract 33 pose landmarks from MMU JSON as time series
    Returns: (num_frames, 33, 2) - x, y coordinates
    """
    with open(json_path, 'r') as f:
        data = json.load(f)

    if len(data) < 30:
        return None

    # Halpe indices for body keypoints (mapped to 17 keypoints)
    body_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
    # Nose, Neck, shoulders, elbows, wrists, hips, knees, ankles

    pose_sequence = []

    for frame in data[:200]:  # Limit to first 200 frames
        keypoints = np.array(frame['keypoints']).reshape(-1, 3)
        frame_pose = []

        for idx in body_indices:
            if idx < len(keypoints) and keypoints[idx][2] > 0.2:
                frame_pose.append([keypoints[idx][0], keypoints[idx][1]])
            else:
                frame_pose.append([0.0, 0.0])

        pose_sequence.append(np.array(frame_pose))

    if len(pose_sequence) < 30:
        return None

    # Normalize to fixed length
    pose_sequence = np.array(pose_sequence)

    # Resize to sequence_length
    if len(pose_sequence) > sequence_length:
        indices = np.linspace(0, len(pose_sequence)-1, sequence_length, dtype=int)
        pose_sequence = pose_sequence[indices]
    else:
        pad_length = sequence_length - len(pose_sequence)
        pose_sequence = np.vstack([pose_sequence, np.zeros((pad_length, 17, 2))])

    return pose_sequence

# Test extraction
test_file = DATASET_PATH + "NORMAL/normal_pexels_03032022_2.json"
pose_seq = extract_pose_sequence(test_file)

if pose_seq is not None:
    print(f"✅ Pose sequence extracted: {pose_seq.shape}")
    print(f"   Frames: {pose_seq.shape[0]}, Keypoints: {pose_seq.shape[1]}, Coordinates: {pose_seq.shape[2]}")
else:
    print("❌ Failed")

EXTRACTING POSE LANDMARKS (33 KEYPOINTS) FROM MMU
✅ Pose sequence extracted: (50, 17, 2)
   Frames: 50, Keypoints: 17, Coordinates: 2


In [14]:
print("="*60)
print("PROCESSING ALL MMU FILES - POSE SEQUENCES")
print("="*60)

categories = ['NORMAL', 'MILD', 'MODERATE', 'SEVERE']
X_sequences = []
y_labels = []

for category in categories:
    folder_path = os.path.join(DATASET_PATH, category)
    if not os.path.exists(folder_path):
        continue

    label = 0 if category == 'NORMAL' else 1
    json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]

    print(f"\n{category}: {len(json_files)} files")

    success_count = 0
    for json_file in json_files:
        try:
            json_path = os.path.join(folder_path, json_file)
            pose_seq = extract_pose_sequence(json_path)
            if pose_seq is not None:
                X_sequences.append(pose_seq)
                y_labels.append(label)
                success_count += 1
        except Exception as e:
            pass

    print(f"  ✅ Processed: {success_count} files")

X_sequences = np.array(X_sequences)
y_labels = np.array(y_labels)

print(f"\n" + "="*60)
print(f"✅ Total samples: {len(X_sequences)}")
print(f"   Control: {sum(y_labels == 0)}")
print(f"   PD: {sum(y_labels == 1)}")
print(f"   Sequence shape: {X_sequences.shape}")

PROCESSING ALL MMU FILES - POSE SEQUENCES

NORMAL: 150 files
  ✅ Processed: 150 files

MILD: 29 files
  ✅ Processed: 29 files

MODERATE: 61 files
  ✅ Processed: 61 files

SEVERE: 52 files
  ✅ Processed: 52 files

✅ Total samples: 292
   Control: 150
   PD: 142
   Sequence shape: (292, 50, 17, 2)


# **Below cell gave 86% accuracy but it's random**

In [17]:
print("="*60)
print("TRAINING LSTM ON POSE SEQUENCES")
print("="*60)

# Reshape for LSTM: (samples, timesteps, features)
# We have: (292, 50, 17, 2) → flatten keypoints to (292, 50, 34)
X = X_sequences.reshape(X_sequences.shape[0], X_sequences.shape[1], -1)
print(f"Input shape: {X.shape}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# Build LSTM model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(50, 34)),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(model.summary())

# Train
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1
)

# Evaluate
y_pred = (model.predict(X_test) > 0.5).astype(int)
acc = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {acc:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Control', 'PD']))

TRAINING LSTM ON POSE SEQUENCES
Input shape: (292, 50, 34)
Train: 233, Test: 59


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 50, 64)         │        25,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 50, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,305 (149.63 KB)

 Trainable params: 38,305 (149.63 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 10s 82ms/step - accuracy: 0.5279 - loss: 0.6928 - val_accuracy: 0.6271 - val_loss: 0.6653
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.5579 - loss: 0.6737 - val_accuracy: 0.6780 - val_loss: 0.6529
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.6266 - loss: 0.6565 - val_accuracy: 0.7288 - val_loss: 0.6286
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.6824 - loss: 0.6209 - val_accuracy: 0.7119 - val_loss: 0.6199
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.7082 - loss: 0.5993 - val_accuracy: 0.7797 - val_loss: 0.5599
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.7210 - loss: 0.5612 - val_accuracy: 0.7288 - val_loss: 0.5423
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.7210 - loss: 0.5599 - val_accuracy: 0.6949 - val_loss: 0.5618
Epoch 8/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.7597 - loss: 0.5036 - val_accuracy: 0.74

In [19]:
print("="*60)
print("SAVING LSTM MODEL")
print("="*60)

# Save model
model_path = MODELS_PATH + "lstm_gait_model.keras"
model.save(model_path)
print(f"✅ Model saved to: {model_path}")

# Also save as .h5 for compatibility
model.save(MODELS_PATH + "lstm_gait_model.h5")

print("\n" + "="*60)
print("✅ LSTM MODEL COMPLETE!")
print("="*60)
print(f"Test Accuracy: {acc:.3f}")
print(f"Samples: {len(X_sequences)}")
print(f"Sequence length: {X_sequences.shape[1]} frames")
print(f"Keypoints: {X_sequences.shape[2]}")

SAVING LSTM MODEL
✅ Model saved to: /content/drive/MyDrive/NeuroSense AI 2/models mmu/lstm_gait_model.keras

✅ LSTM MODEL COMPLETE!
Test Accuracy: 0.864
Samples: 292
Sequence length: 50 frames
Keypoints: 17


# **Below cell will give consistent result**

In [21]:
print("="*60)
print("DETERMINISTIC LSTM (REPRODUCIBLE)")
print("="*60)

import tensorflow as tf
import numpy as np
import random

# 1. FIX ALL SEEDS
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    # For TensorFlow 2.x
    tf.keras.utils.set_random_seed(seed)
    # For reproducibility
    tf.config.experimental.enable_op_determinism()

set_seeds(42)

# 2. FIX DATA SPLIT
from sklearn.model_selection import train_test_split

X = X_sequences.reshape(X_sequences.shape[0], X_sequences.shape[1], -1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# 3. FIX LSTM ARCHITECTURE
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(50, 34), kernel_initializer='he_normal'),
    Dropout(0.3),
    LSTM(32, kernel_initializer='he_normal'),
    Dropout(0.3),
    Dense(16, activation='relu', kernel_initializer='he_normal'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(model.summary())

# 4. TRAIN WITH FIXED PARAMETERS
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[
        EarlyStopping(patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
    ],
    verbose=1,
    shuffle=True,  # KEEP shuffling for better training
    validation_split=0  # Use fixed validation set
)

# 5. EVALUATE
y_pred = (model.predict(X_test) > 0.5).astype(int)
acc = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {acc:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Control', 'PD']))

# 6. SAVE
model.save(MODELS_PATH + "lstm_gait_fixed.keras")
print(f"\n✅ Model saved to: {MODELS_PATH}lstm_gait_fixed.keras")

DETERMINISTIC LSTM (REPRODUCIBLE)
Train: 233, Test: 59


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                   │ (None, 50, 64)         │        25,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 50, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,305 (149.63 KB)

 Trainable params: 38,305 (149.63 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 12s 82ms/step - accuracy: 0.5322 - loss: 0.6953 - val_accuracy: 0.6271 - val_loss: 0.6711 - learning_rate: 0.0010
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.5365 - loss: 0.6889 - val_accuracy: 0.6102 - val_loss: 0.6747 - learning_rate: 0.0010
Epoch 3/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.5536 - loss: 0.6744 - val_accuracy: 0.5932 - val_loss: 0.6708 - learning_rate: 0.0010
Epoch 4/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.6352 - loss: 0.6636 - val_accuracy: 0.6610 - val_loss: 0.6576 - learning_rate: 0.0010
Epoch 5/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.6481 - loss: 0.6457 - val_accuracy: 0.6610 - val_loss: 0.6412 - learning_rate: 0.0010
Epoch 6/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.6910 - loss: 0.6335 - val_accuracy: 0.6610 - val_loss: 0.6247 - learning_rate: 0.0010
Epoch 7/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.6438 - loss: 0.6326 - v

In [23]:
print("="*60)
print("TESTING LSTM 83% fixed ON YOUR VIDEOS")
print("="*60)

# Load the saved model
from tensorflow.keras.models import load_model

model = load_model(MODELS_PATH + "lstm_gait_fixed.keras")
print("✅ Model loaded!")

# Function to extract pose sequence from video
def extract_pose_from_video(video_path, sequence_length=50):
    """Extract pose sequence from a video file using MediaPipe"""
    import cv2
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision

    # Download model if not present
    model_path = 'pose_landmarker_lite.task'
    if not os.path.exists(model_path):
        import urllib.request
        url = 'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task'
        urllib.request.urlretrieve(url, model_path)

    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        output_segmentation_masks=False,
        running_mode=vision.RunningMode.VIDEO
    )
    detector = vision.PoseLandmarker.create_from_options(options)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 30

    pose_sequence = []
    frame_count = 0

    # Body keypoint indices to extract (17 keypoints)
    body_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        detection_result = detector.detect_for_video(mp_image, int(frame_count * 1000 / fps))

        if detection_result.pose_landmarks:
            landmarks = detection_result.pose_landmarks[0]
            h, w, _ = frame.shape
            frame_pose = []

            # Extract 17 keypoints
            for idx in body_indices:
                if idx < len(landmarks):
                    frame_pose.append([landmarks[idx].x * w, landmarks[idx].y * h])
                else:
                    frame_pose.append([0.0, 0.0])

            pose_sequence.append(np.array(frame_pose))

        frame_count += 1
        if frame_count > 500:
            break

    cap.release()
    detector.close()

    if len(pose_sequence) < 30:
        return None

    pose_sequence = np.array(pose_sequence)

    # Resize to sequence_length
    if len(pose_sequence) > sequence_length:
        indices = np.linspace(0, len(pose_sequence)-1, sequence_length, dtype=int)
        pose_sequence = pose_sequence[indices]
    else:
        pad_length = sequence_length - len(pose_sequence)
        pose_sequence = np.vstack([pose_sequence, np.zeros((pad_length, 17, 2))])

    # Flatten: (50, 17, 2) → (50, 34)
    return pose_sequence.reshape(1, sequence_length, -1)

# Test on your videos
VIDEOS_PATH = PROJECT_PATH + "videos/"

test_videos = [
    ("Normal Gait 1.mp4", "Control"),
    ("Normal Gait 2.mp4", "Control"),
    ("Normal Gait 3.mp4", "Control"),
    ("Normal Gait 4.mp4", "Control"),
    ("Normal Gait 5.mp4", "Control"),
    ("Abnormal Gait 1.mp4", "PD"),
    ("Abnormal Gait 2.mp4", "PD"),
    ("Abnormal Gait 3.mp4", "PD"),
    ("Abnormal Gait 4.mp4", "PD"),
    ("Abnormal Gait 5.mp4", "PD"),
    ("Abnormal Gait 6.mp4", "PD"),

]

print("\nTesting on your videos:")
print("-"*40)

for video_name, actual in test_videos:
    video_path = VIDEOS_PATH + video_name
    if os.path.exists(video_path):
        X_video = extract_pose_from_video(video_path)
        if X_video is not None:
            pred_prob = model.predict(X_video, verbose=0)[0][0]
            pred = "PD" if pred_prob > 0.5 else "Control"
            status = "✅" if pred == actual else "❌"
            print(f"{video_name}: {status} Actual: {actual} → Prediction: {pred} (PD prob: {pred_prob:.3f})")
        else:
            print(f"{video_name}: ❌ Could not extract pose")
    else:
        print(f"{video_name}: File not found")

TESTING LSTM ON YOUR VIDEOS
✅ Model loaded!

Testing on your videos:
----------------------------------------
Normal Gait 1.mp4: ❌ Actual: Control → Prediction: PD (PD prob: 0.777)
Normal Gait 2.mp4: ❌ Actual: Control → Prediction: PD (PD prob: 0.723)
Normal Gait 3.mp4: ❌ Actual: Control → Prediction: PD (PD prob: 0.767)
Normal Gait 4.mp4: ❌ Actual: Control → Prediction: PD (PD prob: 0.795)
Normal Gait 5.mp4: ❌ Actual: Control → Prediction: PD (PD prob: 0.744)
Abnormal Gait 1.mp4: ✅ Actual: PD → Prediction: PD (PD prob: 0.681)
Abnormal Gait 2.mp4: ✅ Actual: PD → Prediction: PD (PD prob: 0.777)
Abnormal Gait 3.mp4: ✅ Actual: PD → Prediction: PD (PD prob: 0.745)
Abnormal Gait 4.mp4: ✅ Actual: PD → Prediction: PD (PD prob: 0.821)
Abnormal Gait 5.mp4: ✅ Actual: PD → Prediction: PD (PD prob: 0.691)
Abnormal Gait 6.mp4: ✅ Actual: PD → Prediction: PD (PD prob: 0.805)


In [24]:
print("="*60)
print("TESTING LSTM 86% random ON YOUR VIDEOS")
print("="*60)

# Load the saved model
from tensorflow.keras.models import load_model

model = load_model(MODELS_PATH + "lstm_gait_model.keras")
print("✅ Model loaded!")

# Function to extract pose sequence from video
def extract_pose_from_video(video_path, sequence_length=50):
    """Extract pose sequence from a video file using MediaPipe"""
    import cv2
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision

    # Download model if not present
    model_path = 'pose_landmarker_lite.task'
    if not os.path.exists(model_path):
        import urllib.request
        url = 'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task'
        urllib.request.urlretrieve(url, model_path)

    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        output_segmentation_masks=False,
        running_mode=vision.RunningMode.VIDEO
    )
    detector = vision.PoseLandmarker.create_from_options(options)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 30

    pose_sequence = []
    frame_count = 0

    # Body keypoint indices to extract (17 keypoints)
    body_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        detection_result = detector.detect_for_video(mp_image, int(frame_count * 1000 / fps))

        if detection_result.pose_landmarks:
            landmarks = detection_result.pose_landmarks[0]
            h, w, _ = frame.shape
            frame_pose = []

            # Extract 17 keypoints
            for idx in body_indices:
                if idx < len(landmarks):
                    frame_pose.append([landmarks[idx].x * w, landmarks[idx].y * h])
                else:
                    frame_pose.append([0.0, 0.0])

            pose_sequence.append(np.array(frame_pose))

        frame_count += 1
        if frame_count > 500:
            break

    cap.release()
    detector.close()

    if len(pose_sequence) < 30:
        return None

    pose_sequence = np.array(pose_sequence)

    # Resize to sequence_length
    if len(pose_sequence) > sequence_length:
        indices = np.linspace(0, len(pose_sequence)-1, sequence_length, dtype=int)
        pose_sequence = pose_sequence[indices]
    else:
        pad_length = sequence_length - len(pose_sequence)
        pose_sequence = np.vstack([pose_sequence, np.zeros((pad_length, 17, 2))])

    # Flatten: (50, 17, 2) → (50, 34)
    return pose_sequence.reshape(1, sequence_length, -1)

# Test on your videos
VIDEOS_PATH = PROJECT_PATH + "videos/"

test_videos = [
    ("Normal Gait 1.mp4", "Control"),
    ("Normal Gait 2.mp4", "Control"),
    ("Normal Gait 3.mp4", "Control"),
    ("Normal Gait 4.mp4", "Control"),
    ("Normal Gait 5.mp4", "Control"),
    ("Abnormal Gait 1.mp4", "PD"),
    ("Abnormal Gait 2.mp4", "PD"),
    ("Abnormal Gait 3.mp4", "PD"),
    ("Abnormal Gait 4.mp4", "PD"),
    ("Abnormal Gait 5.mp4", "PD"),
    ("Abnormal Gait 6.mp4", "PD"),

]

print("\nTesting on your videos:")
print("-"*40)

for video_name, actual in test_videos:
    video_path = VIDEOS_PATH + video_name
    if os.path.exists(video_path):
        X_video = extract_pose_from_video(video_path)
        if X_video is not None:
            pred_prob = model.predict(X_video, verbose=0)[0][0]
            pred = "PD" if pred_prob > 0.5 else "Control"
            status = "✅" if pred == actual else "❌"
            print(f"{video_name}: {status} Actual: {actual} → Prediction: {pred} (PD prob: {pred_prob:.3f})")
        else:
            print(f"{video_name}: ❌ Could not extract pose")
    else:
        print(f"{video_name}: File not found")

TESTING LSTM ON YOUR VIDEOS
✅ Model loaded!

Testing on your videos:
----------------------------------------
Normal Gait 1.mp4: ✅ Actual: Control → Prediction: Control (PD prob: 0.120)
Normal Gait 2.mp4: ✅ Actual: Control → Prediction: Control (PD prob: 0.228)
Normal Gait 3.mp4: ✅ Actual: Control → Prediction: Control (PD prob: 0.171)
Normal Gait 4.mp4: ❌ Actual: Control → Prediction: PD (PD prob: 0.532)
Normal Gait 5.mp4: ✅ Actual: Control → Prediction: Control (PD prob: 0.365)
Abnormal Gait 1.mp4: ❌ Actual: PD → Prediction: Control (PD prob: 0.370)
Abnormal Gait 2.mp4: ✅ Actual: PD → Prediction: PD (PD prob: 0.669)
Abnormal Gait 3.mp4: ✅ Actual: PD → Prediction: PD (PD prob: 0.505)
Abnormal Gait 4.mp4: ✅ Actual: PD → Prediction: PD (PD prob: 0.682)
Abnormal Gait 5.mp4: ✅ Actual: PD → Prediction: PD (PD prob: 0.572)
Abnormal Gait 6.mp4: ❌ Actual: PD → Prediction: Control (PD prob: 0.063)
